# 🎬 Simple AI Video Generator

Generate animated videos from your character image using Stable Video Diffusion.

**This actually works in Google Colab!**

1. Click **Runtime → Run all**
2. Upload your character image
3. Upload your audio
4. Wait 5-10 minutes
5. Download result

In [ ]:
# Install minimal dependencies
!pip install -q diffusers[torch] transformers accelerate
!pip install -q imageio[ffmpeg] moviepy
print("✓ Installed")

In [ ]:
# Upload files
from google.colab import files

print("📸 Upload CHARACTER IMAGE:")
uploaded = files.upload()
image_file = list(uploaded.keys())[0]

print("\n🎵 Upload AUDIO:")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

print("\n✓ Files uploaded")

In [ ]:
# Generate video from image
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from PIL import Image

print("🎬 Loading model...")

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.enable_model_cpu_offload()

print("✓ Model loaded")
print("\n🎨 Generating video (5-10 minutes)...")

# Load and prepare image
image = Image.open(image_file)
image = image.resize((1024, 576))

# Generate video
generator = torch.manual_seed(42)
frames = pipe(image, decode_chunk_size=8, generator=generator).frames[0]

export_to_video(frames, "generated_video.mp4", fps=7)

print("✓ Video generated!")

In [ ]:
# Add audio
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips

print("🎵 Adding audio...")

video = VideoFileClip("generated_video.mp4")
audio = AudioFileClip(audio_file)

# Loop video to match audio length
if audio.duration > video.duration:
    loops_needed = int(audio.duration / video.duration) + 1
    video = concatenate_videoclips([video] * loops_needed)
    video = video.subclip(0, audio.duration)

final = video.set_audio(audio)
final.write_videofile("final_video.mp4", codec="libx264", audio_codec="aac", logger=None)

print("✓ Done!")

In [ ]:
# Preview and download
from IPython.display import Video
from google.colab import files

print("📺 Preview:")
display(Video("final_video.mp4", width=640))

print("\n📥 Downloading...")
files.download("final_video.mp4")

print("\n✅ COMPLETE! Your animated video is ready.")